In [1]:
import cobra
import pandas
from cobra.io import read_sbml_model, write_sbml_model
import logging
from cobra.flux_analysis import flux_variability_analysis
from cobra.flux_analysis import gapfill
from cobra.flux_analysis.loopless import add_loopless, loopless_solution
from cobra import Model, Reaction, Metabolite
from copy import deepcopy
from collections import defaultdict
from cobra.io import load_json_model

In [2]:
#Function based on https://github.com/opencobra/cobrapy/issues/707 and completely altered by me!

def removeDuplicateRxn(model):
    model2 = deepcopy(model)
    toRemove = []
    doubt = []
    
    for eachReaction in model.reactions:

        if(eachReaction.id not in toRemove):

            ids = []
            stechiometry = []
            gn = []
        
            #Placing metabolites and genes of R1 in list
            for eachMet in eachReaction.metabolites:
                ids.append(eachMet.id)
                stechiometry.append(eachReaction.metabolites[eachMet])

            ids.sort()
        
            for eachGene in eachReaction.genes:
                gn.append(eachGene.id)
    
        #Starting comparison
            for eachReaction2 in model2.reactions:
            
                if(eachReaction2.id not in toRemove):
                           
                    if eachReaction.id != eachReaction2.id: #Comparing ids to avoid self comparison
                
                        ids2=[]
                        stechiometry2 = []

                        for eachMet2 in eachReaction2.metabolites:
                            ids2.append(eachMet2.id)
                            stechiometry2.append(eachReaction2.metabolites[eachMet2])
                
                        ids2.sort()    
                
                        if(ids == ids2): #all metabolites are the same
                    
                            #Comparing genes
                            duplicate = 0
                            unsure = 0
                        
                            if(len(gn) == 0 or len(eachReaction2.genes) == 0): #one of the reactions don't have associated genes
                                unsure = 1

                            else:
                                for eachGene2 in eachReaction2.genes:
                                    if(eachGene2.id in gn): #At least one gene is the same
                                        duplicate = 1
                                        break
                        
                            if(duplicate == 1): #Checking if any reaction is reversible and removing the non-reversible one
                                if(eachReaction.lower_bound != 0 and eachReaction.upper_bound != 0): 
                                    toRemove.append(eachReaction2.id) 
                                elif(eachReaction2.lower_bound != 0 and eachReaction2.upper_bound != 0):
                                    toRemove.append(eachReaction.id)
                                else:
                                    unsure = 1
                                    
                            if(unsure == 1):
                                td=[]
                                td.append(eachReaction.id)
                                td.append(eachReaction2.id)
                                td.sort()
                                doubt.append(str(td[0]+":"+td[1]))
                                
                
    a = list(set(doubt))
    model2.remove_reactions(toRemove)
    model2.repair()
    return[model2,toRemove,a]

In [3]:
#Changing configuration
cobra_config = cobra.Configuration()
cobra_config.bounds = -999999.0,999999.0
cobra_config.solver = "cplex"

In [4]:
#Setting inputfile
input = "/scr/k61san/natasha/matomic/trials/CarveMe/Bproducta.tcds.top4.gramPosN.cim8.xml"

In [5]:
#Read carveme model
model = cobra.io.read_sbml_model(str(input))

In [6]:
#Fixing masses
model.metabolites.get_by_id("ACP_c").formula = "C11H21N2O7PRS"
model.metabolites.get_by_id("arachACP_c").formula = "C31H59N2O8PRS"
model.metabolites.get_by_id("benzcoa_c").formula = "C28H36N7O17P3S"
model.metabolites.get_by_id("ddcaACP_c").formula = "C23H43N2O8PRS"
model.metabolites.get_by_id("dmso2_c").formula = "C2H6O2S"
model.metabolites.get_by_id("dmso2_e").formula = "C2H6O2S"
model.metabolites.get_by_id("dmso2_p").formula = "C2H6O2S"
model.metabolites.get_by_id("fad_c").formula = "C27H31N9O15P2"
model.metabolites.get_by_id("fadh2_c").formula = "C27H33N9O15P2"
model.metabolites.get_by_id("fmn_c").formula = "C17H19N4O9P"
model.metabolites.get_by_id("fmnh2_c").formula = "C17H21N4O9P"
model.metabolites.get_by_id("hdeACP_c").formula = "C27H49N2O8PRS"
model.metabolites.get_by_id("Nforglu_c").formula = "C6H7NO5"
model.metabolites.get_by_id("ocdcaACP_c").formula = "C29H55N2O8PRS"
model.metabolites.get_by_id("octeACP_c").formula = "C29H53N2O8PRS"
model.metabolites.get_by_id("palmACP_c").formula = "C27H51N2O8PRS"
model.metabolites.get_by_id("pdima_c").formula = "C96H190O5"
model.metabolites.get_by_id("pdima_e").formula = "C96H190O5"
model.metabolites.get_by_id("pea_c").formula = "C8H10O"
model.metabolites.get_by_id("pea_e").formula = "C8H10O"
model.metabolites.get_by_id("phthdl_c").formula = "C31H62O3"
model.metabolites.get_by_id("phthiocerol_c").formula = "C32H66O3"
model.metabolites.get_by_id("prephth_c").formula = "C32H61O5"
model.metabolites.get_by_id("prephthACP_c").formula = "C43H81N2O11PRS"
model.metabolites.get_by_id("ribflv_c").formula = "C17H20N4O6"
model.metabolites.get_by_id("ribflv_e").formula = "C17H20N4O6"
model.metabolites.get_by_id("tdeACP_c").formula = "C25H45N2O8PRS"

In [7]:
#Fixing charges
model.metabolites.get_by_id("2ahbut_c").charge = -1
model.metabolites.get_by_id("2dr1p_c").charge = -2
model.metabolites.get_by_id("2maacoa_c").charge = -4
model.metabolites.get_by_id("2me4p_c").charge = -2
model.metabolites.get_by_id("2o3mpt_c").charge = -1
model.metabolites.get_by_id("2shchc_c").charge = -2
model.metabolites.get_by_id("3hocoa_c").charge = -4
model.metabolites.get_by_id("3uib_c").charge = -1
model.metabolites.get_by_id("5aizc_c").charge = -3
model.metabolites.get_by_id("6pgg_c").charge = -2
model.metabolites.get_by_id("acgam1p_c").charge = -2
model.metabolites.get_by_id("acmanap_c").charge = -2
model.metabolites.get_by_id("ACP_c").charge = -1
model.metabolites.get_by_id("air_c").charge = -2
model.metabolites.get_by_id("anhgm3p_p").charge = -2
model.metabolites.get_by_id("argsuc_c").charge = -1
model.metabolites.get_by_id("dcamp_c").charge = -4
model.metabolites.get_by_id("ddcaACP_c").charge = -1
model.metabolites.get_by_id("dhpmp_c").charge = -2
model.metabolites.get_by_id("dscl_c").charge = -7
model.metabolites.get_by_id("fad_c").charge = -2
model.metabolites.get_by_id("fadh2_c").charge = -2
model.metabolites.get_by_id("fc1p_c").charge = -2
model.metabolites.get_by_id("fdp_c").charge = -4
model.metabolites.get_by_id("fdxrd_c").charge = 0
model.metabolites.get_by_id("fgam_c").charge = -2
model.metabolites.get_by_id("fmn_c").charge = -2
model.metabolites.get_by_id("fmnh2_c").charge = -2
model.metabolites.get_by_id("fpram_c").charge = -2 #nao mudar
model.metabolites.get_by_id("frulysp_c").charge = -1
model.metabolites.get_by_id("fruur_c").charge = -1
model.metabolites.get_by_id("g3p_c").charge = -2
model.metabolites.get_by_id("g3pg_c").charge = -1
model.metabolites.get_by_id("g3pg_e").charge = -1
model.metabolites.get_by_id("g3pg_p").charge = -1
model.metabolites.get_by_id("gdptp_c").charge = -7
model.metabolites.get_by_id("man1p_c").charge = -2
model.metabolites.get_by_id("man6p_c").charge = -2
model.metabolites.get_by_id("man6pglyc_c").charge = -3
model.metabolites.get_by_id("man6pglyc_e").charge = -3
model.metabolites.get_by_id("murein4px4p_p").charge = -4
model.metabolites.get_by_id("murein5px4p_p").charge = -4
model.metabolites.get_by_id("murein5px4px4p_p").charge = -6
model.metabolites.get_by_id("Nforglu_c").charge = -2
model.metabolites.get_by_id("octeACP_c").charge = -1
model.metabolites.get_by_id("pa120_c").charge = -2
model.metabolites.get_by_id("pa120_p").charge = -2
model.metabolites.get_by_id("pa161_c").charge = -2
model.metabolites.get_by_id("pa161_p").charge = -2
model.metabolites.get_by_id("pa180_c").charge = -2
model.metabolites.get_by_id("pa180_p").charge = -2
model.metabolites.get_by_id("pa181_c").charge = -2
model.metabolites.get_by_id("pa181_p").charge = -2
model.metabolites.get_by_id("Pald_c").charge = -2
model.metabolites.get_by_id("palmACP_c").charge = -1
model.metabolites.get_by_id("peptido_BS_c").charge = -2
model.metabolites.get_by_id("pg120_c").charge = -1
model.metabolites.get_by_id("pg120_p").charge = -1
model.metabolites.get_by_id("pg160_c").charge = -1
model.metabolites.get_by_id("pg160_p").charge = -1
model.metabolites.get_by_id("pg161_c").charge = -1
model.metabolites.get_by_id("pg161_p").charge = -1
model.metabolites.get_by_id("pg180_c").charge = -1
model.metabolites.get_by_id("pg180_p").charge = -1
model.metabolites.get_by_id("pgp120_c").charge = -3
model.metabolites.get_by_id("pgp120_p").charge = -3
model.metabolites.get_by_id("pgp160_c").charge = -3
model.metabolites.get_by_id("pgp160_p").charge = -3
model.metabolites.get_by_id("pgp161_c").charge = -3
model.metabolites.get_by_id("pgp161_p").charge = -3
model.metabolites.get_by_id("pgp180_c").charge = -3
model.metabolites.get_by_id("pgp180_p").charge = -3
model.metabolites.get_by_id("pgp181_c").charge = -3
model.metabolites.get_by_id("pgp181_p").charge = -3
model.metabolites.get_by_id("ppgpp_c").charge = -6
model.metabolites.get_by_id("pphn_c").charge = -2
model.metabolites.get_by_id("pppi_c").charge = -4
model.metabolites.get_by_id("pqq_p").charge = -3
model.metabolites.get_by_id("pqqh2_p").charge = -3
model.metabolites.get_by_id("prbamp_c").charge = -4
model.metabolites.get_by_id("prbatp_c").charge = -6
model.metabolites.get_by_id("prephthACP_c").charge = -1
model.metabolites.get_by_id("ps120_c").charge = -1
model.metabolites.get_by_id("ps160_c").charge = -1
model.metabolites.get_by_id("ps161_c").charge = -1
model.metabolites.get_by_id("ps180_c").charge = -1
model.metabolites.get_by_id("ps181_c").charge = -1
model.metabolites.get_by_id("r5p_c").charge = -2
model.metabolites.get_by_id("rml1p_c").charge = -2
model.metabolites.get_by_id("sbzcoa_c").charge = -5
model.metabolites.get_by_id("scl_c").charge = -7
model.metabolites.get_by_id("suc6p_c").charge = -2
model.metabolites.get_by_id("tag6p__D_c").charge = -2
model.metabolites.get_by_id("tagdp__D_c").charge = -4
model.metabolites.get_by_id("tagur_c").charge = -1
model.metabolites.get_by_id("tdecoa_c").charge = -4
model.metabolites.get_by_id("trnaglu_c").charge = 0
model.metabolites.get_by_id("tsul_c").charge = -2
model.metabolites.get_by_id("tsul_p").charge = -2
model.metabolites.get_by_id("uaagmda_c").charge = -4
model.metabolites.get_by_id("uagmda_c").charge = -4
model.metabolites.get_by_id("uamr_c").charge = -3
model.metabolites.get_by_id("udcpdp_c").charge = -3
model.metabolites.get_by_id("udcpdp_e").charge = -3
model.metabolites.get_by_id("udcpp_c").charge = -2
model.metabolites.get_by_id("udcpp_e").charge = -2

In [8]:
# Identifying and removing duplicated reactions
# md model with removed reactions
# rd removed list
# dt reactions in doubt

[md,rd, dt] = removeDuplicateRxn(model)

In [9]:
#Checking the ones in doubt
print(dt)

['PRAIS_1:PRFGCL', 'HACD7:HACD7i', 'EX_glcn__D_e:EX_glcn_e', 'MN2tipp:MN2tpp', 'GLYK:GLYK1', 'HACD6:HACD6i', 'HACD4:HACD4i', 'EX_eths_e:EX_ethso3_e', 'GLBRAN2:GLDBRAN2', 'ACTNabc:ACTNabc1', 'GTHRDabc2pp:GTHRDabcpp', 'EX_sula_e:EX_sulfac_e', 'PRAIS:PRAIS_1', 'NP1:NP1_1', 'HACD3:HACD3i', 'EX_abt__L_e:EX_abt_e', 'EX_isetac_e:EX_istnt_e', 'EX_metox_e:EX_metsox_S__L_e', 'HACD2:HACD2i', 'EX_orn__L_e:EX_orn_e', 'COBALT2abcpp:Cobalt2abcppI', 'EX_galct__D_e:EX_galctr__D_e', 'PSUDS:YUMPS', 'ATPM:NTP1', 'HACD5:HACD5i', 'PRAIS:PRFGCL', 'RIBabc:RIBabc1']


In [10]:
#Removing reactions in doubt after manual inspection
md.remove_reactions([md.reactions.get_by_id("EX_abt__L_e"),md.reactions.get_by_id("EX_isetac_e"),
                     md.reactions.get_by_id("EX_glcn__D_e"),md.reactions.get_by_id("EX_ethso3_e"),
                     md.reactions.get_by_id("EX_galctr__D_e"),md.reactions.get_by_id("EX_sulfac_e"),
                     md.reactions.get_by_id("EX_orn__L_e"),md.reactions.get_by_id("EX_metsox_S__L_e"),
                     md.reactions.get_by_id("PSUDS"),md.reactions.get_by_id("PRFGCL"),
                     md.reactions.get_by_id("PRAIS_1"),md.reactions.get_by_id("HACD6i"),
                     md.reactions.get_by_id("HACD4i"),md.reactions.get_by_id("GLYK1"),
                     md.reactions.get_by_id("HACD5i"),md.reactions.get_by_id("HACD7i"),
                     md.reactions.get_by_id("NP1"),md.reactions.get_by_id("HACD2i"),
                     md.reactions.get_by_id("HACD3i")])

md.repair()

In [11]:
#Running FVA to id blocked reactions
#Universally blocked reactions are reactions that during Flux Variability Analysis cannot carry any flux while all 
#model boundaries are open. Generally blocked reactions are caused by network gaps, which can be attributed to 
#scope or knowledge gaps. 
md.summary(fva=0.95)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
arab__L_e,EX_arab__L_e,10,[0; 10],5,28.11%
arg__L_e,EX_arg__L_e,0.1161,[-0.3527; 0.1163],6,0.39%
asn__L_e,EX_asn__L_e,0.09458,[0; 1.244],4,0.21%
asp__L_e,EX_asp__L_e,0.6243,[0; 1.686],4,1.40%
ca2_e,EX_ca2_e,0.002042,[0.00194; 0.002042],0,0.00%
cl_e,EX_cl_e,0.002042,[0.00194; 0.002042],0,0.00%
cobalt2_e,EX_cobalt2_e,3.924E-05,[3.728E-05; 3.767E-05],0,0.00%
cu2_e,EX_cu2_e,0.0002782,[0.0002643; 0.0002782],0,0.00%
cys__L_e,EX_cys__L_e,0.03618,[-0.742; 1.2],3,0.06%
fe2_e,EX_fe2_e,0.01888,[0; 0.09807],0,0.00%


In [12]:
#Loading BiGG's universal model for gapfilling

universal = load_json_model("/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/carveme/data/generated/universal_model_cobrapy.json")

In [13]:
#Performing gapfill with universal model from BiGG to try to reduce blocked reactions
#Usually, it does not do anything (and it takes some time to run)

gapfill(md, universal, exchange_reactions=True, demand_reactions=False, iterations=10)

[[], [], [], [], [], [], [], [], [], []]

In [14]:
#Adding/Removing/Editing reactions manually to reduce blocked reactions

md.remove_reactions([md.reactions.get_by_id("AALDH"),md.reactions.get_by_id("PEAt"), #Not connected to the network
                     md.reactions.get_by_id("PEALtabc"),md.reactions.get_by_id("EX_pea_e")])
md.remove_metabolites([md.metabolites.get_by_id("pacald_c"),md.metabolites.get_by_id("pea_c"),
                       md.metabolites.get_by_id("pea_e")])

md.remove_reactions([md.reactions.get_by_id("BTDD_RR"),md.reactions.get_by_id("ACTD"), #Not connected to the network
                     md.reactions.get_by_id("ACTD_1"), md.reactions.get_by_id("ACTNabc1"),
                     md.reactions.get_by_id("ACTNabc"),md.reactions.get_by_id("ARSR"),md.reactions.get_by_id("SAOR"),
                     md.reactions.get_by_id("EX_actn__R_e")])
md.remove_metabolites([md.metabolites.get_by_id("btd_RR_c"),md.metabolites.get_by_id("actn__R_c"),
                       md.metabolites.get_by_id("actn__R_e"),md.metabolites.get_by_id("actn__S_c"),
                       md.metabolites.get_by_id("diact_c")])

md.remove_reactions([md.reactions.get_by_id("CAt6pp")]) #Useless transport
md.remove_metabolites([md.metabolites.get_by_id("ca2_p")])

md.remove_reactions([md.reactions.get_by_id("HACD2"),md.reactions.get_by_id("HACD3"), #Not connected to the network
                     md.reactions.get_by_id("HACD4"), md.reactions.get_by_id("HACD5"),
                     md.reactions.get_by_id("HACD6"),md.reactions.get_by_id("HACD7")])
md.remove_metabolites([md.metabolites.get_by_id("3ohcoa_c"),md.metabolites.get_by_id("3hhcoa_c"),
                       md.metabolites.get_by_id("3oocoa_c"),md.metabolites.get_by_id("3hocoa_c"),
                       md.metabolites.get_by_id("3odcoa_c"),md.metabolites.get_by_id("3hdcoa_c"),
                       md.metabolites.get_by_id("3oddcoa_c"),md.metabolites.get_by_id("3hddcoa_c"),
                       md.metabolites.get_by_id("3otdcoa_c"),md.metabolites.get_by_id("3htdcoa_c"),
                       md.metabolites.get_by_id("3ohdcoa_c"),md.metabolites.get_by_id("3hhdcoa_c")])

md.remove_reactions([md.reactions.get_by_id("MG2tex")]) #Useless transport
md.remove_metabolites([md.metabolites.get_by_id("mg2_p")])

md.remove_reactions([md.reactions.get_by_id("MS_1")]) #Not connected to the network
md.remove_metabolites([md.metabolites.get_by_id("mhpglu_c"),md.metabolites.get_by_id("hpglu_c")])

md.add_reactions([universal.reactions.get_by_id("ABTt")]) #Adding missing transport

md.remove_reactions([md.reactions.get_by_id("EX_btn_e"),md.reactions.get_by_id("BTNt2i"), #Not connected to the network
                     md.reactions.get_by_id("BTNTe")])
md.remove_metabolites([md.metabolites.get_by_id("btn_e"),md.metabolites.get_by_id("btn_c")])

md.remove_reactions([md.reactions.get_by_id("EX_eths_e"),md.reactions.get_by_id("EX_galct__D_e"),
                     md.reactions.get_by_id("EX_istnt_e"),md.reactions.get_by_id("EX_met__D_e"),
                     md.reactions.get_by_id("METDabc"),md.reactions.get_by_id("METte"),
                     md.reactions.get_by_id("EX_sula_e")]) #Not connected to the network
md.remove_metabolites([md.metabolites.get_by_id("eths_e"),md.metabolites.get_by_id("galct__D_e"),
                       md.metabolites.get_by_id("istnt_e"),md.metabolites.get_by_id("met__D_e"),
                       md.metabolites.get_by_id("met__D_c"),md.metabolites.get_by_id("sula_e")])

md.add_metabolites([universal.metabolites.get_by_id("etoh_e")]) #Adding missing transport
md.metabolites.get_by_id("etoh_e").formula = "C2H6O"
md.metabolites.get_by_id("etoh_e").charge = 0
md.add_reactions([universal.reactions.get_by_id("EX_etoh_e"),universal.reactions.get_by_id("ETOHtex")]) #Adding missing transport

md.repair()

In [15]:
md.summary(fva=0.95)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
arab__L_e,EX_arab__L_e,10,[0; 10],5,28.11%
arg__L_e,EX_arg__L_e,0.1161,[-0.3527; 0.1163],6,0.39%
asn__L_e,EX_asn__L_e,0.09458,[0; 1.244],4,0.21%
asp__L_e,EX_asp__L_e,0.6243,[0; 1.686],4,1.40%
ca2_e,EX_ca2_e,0.002042,[0.00194; 0.002042],0,0.00%
cl_e,EX_cl_e,0.002042,[0.00194; 0.002042],0,0.00%
cobalt2_e,EX_cobalt2_e,3.924E-05,[3.728E-05; 3.767E-05],0,0.00%
cu2_e,EX_cu2_e,0.0002782,[0.0002643; 0.0002782],0,0.00%
cys__L_e,EX_cys__L_e,0.03618,[-0.742; 1.2],3,0.06%
fe2_e,EX_fe2_e,0.01888,[0; 0.09807],0,0.00%


In [16]:
#Loopless FBA to solve Stoichiometrically Balanced Cycles (does not help much)

rlist = [md.reactions.get_by_id("13PPDH2"),md.reactions.get_by_id("13PPDH"),
         md.reactions.get_by_id("2ACLMM"),md.reactions.get_by_id("4ABUTD"),md.reactions.get_by_id("ABUTD"),
         md.reactions.get_by_id("EX_glcn_e"),md.reactions.get_by_id("5DGLCNR"),
         md.reactions.get_by_id("5DKGR"),md.reactions.get_by_id("5DGLCN_Et"),
         md.reactions.get_by_id("IDOND"),md.reactions.get_by_id("IDOND2"),
         md.reactions.get_by_id("ACONT"),md.reactions.get_by_id("ADAPAT"),md.reactions.get_by_id("ADK1"),
         md.reactions.get_by_id("ADK2"),md.reactions.get_by_id("ADKd"),md.reactions.get_by_id("ALAD_L"),
         md.reactions.get_by_id("ALAR"),md.reactions.get_by_id("ALATA_D"),md.reactions.get_by_id("ALATA_L"),
         md.reactions.get_by_id("ALCD19"),md.reactions.get_by_id("ALCD19y"),md.reactions.get_by_id("ALCD4"),
         md.reactions.get_by_id("ACONTa"),md.reactions.get_by_id("ACONTb"),
         md.reactions.get_by_id("ALCD4y"),md.reactions.get_by_id("ARABR"),md.reactions.get_by_id("ARABRr"),
         md.reactions.get_by_id("ASPT"),md.reactions.get_by_id("ASPTA"),md.reactions.get_by_id("BGLA"),
         md.reactions.get_by_id("BTS"),md.reactions.get_by_id("BTS_nadph"),md.reactions.get_by_id("CEPA"),
         md.reactions.get_by_id("CO2t"),md.reactions.get_by_id("CO2tex"),md.reactions.get_by_id("CO2tpp"),
         md.reactions.get_by_id("DADK"),md.reactions.get_by_id("DAPDA"),md.reactions.get_by_id("FUM"),
         md.reactions.get_by_id("G1PP"),md.reactions.get_by_id("G5SADs"),md.reactions.get_by_id("G6PBDH"),
         md.reactions.get_by_id("G6PDH2r"),md.reactions.get_by_id("G6PI"),md.reactions.get_by_id("G6PP"),
         md.reactions.get_by_id("GALM1"),md.reactions.get_by_id("GAPD"),md.reactions.get_by_id("GAPDi_nadp"),
         md.reactions.get_by_id("GK1"),md.reactions.get_by_id("GK2"),md.reactions.get_by_id("GLBRAN2"),
         md.reactions.get_by_id("GLDBRAN2"),md.reactions.get_by_id("GLUDxi"),md.reactions.get_by_id("GLUDy"),
         md.reactions.get_by_id("GLUR"),md.reactions.get_by_id("GalMr"),md.reactions.get_by_id("GalMr_2"),
         md.reactions.get_by_id("H2Ot"),md.reactions.get_by_id("H2Otex"),md.reactions.get_by_id("H2Otpp"),
         md.reactions.get_by_id("HACD1i"),md.reactions.get_by_id("HBCO_nadp"),md.reactions.get_by_id("HOXPRx"),
         md.reactions.get_by_id("HPYRI"),md.reactions.get_by_id("HPYRRx"),md.reactions.get_by_id("KARA1"),
         md.reactions.get_by_id("KARA4"),md.reactions.get_by_id("LCARR"),md.reactions.get_by_id("MDH"),
         md.reactions.get_by_id("MTHFD"),md.reactions.get_by_id("MTHFD2i"),
         md.reactions.get_by_id("MN2tipp"),md.reactions.get_by_id("MN2tpp"),
         md.reactions.get_by_id("NADDP"),md.reactions.get_by_id("NADDPp_1"),md.reactions.get_by_id("NH4t"),
         md.reactions.get_by_id("NH4tex"),md.reactions.get_by_id("NH4tpp"),md.reactions.get_by_id("OCBT"),
         md.reactions.get_by_id("OCBT_1"),md.reactions.get_by_id("ORNCD"),md.reactions.get_by_id("ORNTA"),
         md.reactions.get_by_id("P5CR"),md.reactions.get_by_id("PDBL_3"),md.reactions.get_by_id("PDBL_4"),
         md.reactions.get_by_id("PGCM"),md.reactions.get_by_id("PGI"),md.reactions.get_by_id("PGI1c"),
         md.reactions.get_by_id("PGMT"),md.reactions.get_by_id("PLPS"),md.reactions.get_by_id("PPDOy"),
         md.reactions.get_by_id("PPK2"),md.reactions.get_by_id("PPM2"),md.reactions.get_by_id("PPM2_2"),
         md.reactions.get_by_id("PUNP1"),md.reactions.get_by_id("PUNP1_1"),md.reactions.get_by_id("PUNP2"),
         md.reactions.get_by_id("PUNP2_1"),md.reactions.get_by_id("PUNP3"),md.reactions.get_by_id("PUNP3_1"),
         md.reactions.get_by_id("PUNP4"),md.reactions.get_by_id("PUNP4_1"),md.reactions.get_by_id("PUNP5"),
         md.reactions.get_by_id("PUNP5_1"),md.reactions.get_by_id("PUNP6"),md.reactions.get_by_id("PUNP6_1"),
         md.reactions.get_by_id("PUNP7"),md.reactions.get_by_id("PUNP7_1"),md.reactions.get_by_id("PYDXS"),
         md.reactions.get_by_id("R1PI"),md.reactions.get_by_id("RPI"),md.reactions.get_by_id("SDPDS"),
         md.reactions.get_by_id("SDPTA"),md.reactions.get_by_id("SUCOAACTr"),md.reactions.get_by_id("THDPS"),
         md.reactions.get_by_id("THPAT"),md.reactions.get_by_id("TPI"),md.reactions.get_by_id("TRSARr"),
         md.reactions.get_by_id("VALTA"),md.reactions.get_by_id("VPAMTr")]         
flux_variability_analysis(md,rlist,loopless=True)

/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


,minimum,maximum
13PPDH2,-1000.000000,0.000000
13PPDH,0.000000,0.000000
2ACLMM,0.000000,0.347513
4ABUTD,0.000000,0.000000
ABUTD,0.000000,0.000000
...,...,...
THPAT,0.000000,0.003924
TPI,0.000000,36.074913
TRSARr,0.000000,0.000000
VALTA,-1000.000000,0.000000


In [17]:
output = input.replace("xml", "")
output

'/scr/k61san/natasha/matomic/trials/CarveMe/Bproducta.tcds.top4.gramPosN.cim8.'

In [18]:
#Writting intermediate network
sbml = output + "manual.xml"
print(sbml)
cobra.io.write_sbml_model(md, sbml)

/scr/k61san/natasha/matomic/trials/CarveMe/Bproducta.tcds.top4.gramPosN.cim8.manual.xml
